<a href="https://colab.research.google.com/github/SversusN/LLaMA-LoRA-Tuner/blob/dev/LlamaEsFinetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets peft accelerate bitsandbytes torch sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset

In [ ]:
# Конфигурация квантизации (4-bit)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Загрузка предобученной модели LLaMA 2
model_name = "IlyaGusev/llama_7b_ru_turbo_alpaca_lora_merged"  # Можно использовать LLaMA 3, если доступно
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

# Подготовка модели для k-bit обучения
model = prepare_model_for_kbit_training(model)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/282 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/118 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggin

...

pytorch_model-00001-of-00002.bin:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

pytorch_model-00002-of-00002.bin:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [ ]:
# Загрузка датасета с Hugging Face
dataset = load_dataset("SversusN/es", split="train")

# Проверка первых нескольких записей
print(dataset[:2])

output_1.json:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2797 [00:00<?, ? examples/s]

{'instruction': ['Привяжите товар', 'Привяжите товар'], 'input': ['"умка" колыбельные песенки (книжка-гармошка) *224446', '"фруто няня" 100г. пюре натур. из яблок, груш и персиков "фруктовый салатик" для пит. дет. р/в'], 'output': ['5970978; Игрушка книжка умка колыбельные (224446); Россия; Россия', '4061808; ДП фрутоняня пюре фруктов салат яблоко-груша-персик 100г 5+мес; Прогресс ОАО г.Липецк/Лебедянский ОАО (Липецкая обл); Россия']}


In [ ]:
def format_data(examples):
    # Объединяем инструкцию и вход в один текст
    texts = []
    labels = []
    for instruction, input_text, output in zip(examples["instruction"], examples["input"], examples["output"]):
        text = f"<human>: {instruction} {input_text}\n<bot>: {output}"
        texts.append(text)
        labels.append(output)  # labels — это ожидаемый ответ

    return {"text": texts, "labels": labels}

# Применение форматирования
formatted_dataset = dataset.map(format_data, batched=True)
print(formatted_dataset[0])  # Проверка

Map:   0%|          | 0/127906 [00:00<?, ? examples/s]

{'instructions': 'Привяжите товар', 'input': '752 бандаж-шорты корригир №6 завыш талия черн', 'output': '5355128; Бандаж-шорты корригирующие 752 N6 завыш талия черный; Россия; Россия', 'text': '<human>: Привяжите товар 752 бандаж-шорты корригир №6 завыш талия черн\n<bot>: 5355128; Бандаж-шорты корригирующие 752 N6 завыш талия черный; Россия; Россия', 'labels': '5355128; Бандаж-шорты корригирующие 752 N6 завыш талия черный; Россия; Россия'}


In [ ]:
def tokenize_function(examples):
    # Токенизация текста и labels
    inputs = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)
    labels = tokenizer(examples["labels"], truncation=True, padding="max_length", max_length=512)["input_ids"]

    return {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
        "labels": labels
    }

# Применение токенизации
tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True, remove_columns=["text", "labels"])
print(tokenized_dataset[0])  # Проверка

Map:   0%|          | 0/127906 [00:00<?, ? examples/s]

{'instructions': 'Привяжите товар', 'input': '752 бандаж-шорты корригир №6 завыш талия черн', 'output': '5355128; Бандаж-шорты корригирующие 752 N6 завыш талия черный; Россия; Россия', 'labels': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

То, что ниже надо протестировать.

In [ ]:
  def format_and_tokenize(examples):  # Формируем входной текст (инструкция + вход)

    eos_token = tokenizer.eos_token  # Обязательно добавьте EOS_TOKEN!
    inputs = [f"<human>: {ins} {inp}{eos_token}" for ins, inp in zip(examples["instruction"], examples["input"])]
    # Целевой текст — это ответ
    targets = [f"<bot>: {out}{eos_token}" for out in examples["output"]]

    # Токенизация входных данных
    tokenized_inputs = tokenizer(
        inputs,
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )

    # Токенизация целевых данных (labels)
    tokenized_targets = tokenizer(
        targets,
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )

    # Объединяем и возвращаем
    return {
        "input_ids": tokenized_inputs["input_ids"].tolist(),
        "attention_mask": tokenized_inputs["attention_mask"].tolist(),
        "labels": tokenized_targets["input_ids"].tolist()
    }

# Применение функции
tokenized_dataset = dataset.map(
    format_and_tokenize,
    batched=True,
    remove_columns=["instruction", "input", "output"]
)

# Проверка
print(tokenized_dataset[0])

Map:   0%|          | 0/2797 [00:00<?, ? examples/s]

{'input_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [ ]:
# Конфигурация LoRA
lora_config = LoraConfig(
    r=16,  # Ранг корректировки
    lora_alpha=32,  # Масштабирующий коэффициент
    target_modules=["q_proj", "v_proj"],  # Модули, к которым применяется LoRA
    lora_dropout=0.05,  # Dropout для LoRA
    bias="none",
    task_type="CAUSAL_LM"
)

# Применение LoRA к модели
model = get_peft_model(model, lora_config)
print(model.print_trainable_parameters())  # Проверка количества обучаемых параметров

trainable params: 8,388,608 || all params: 6,746,804,224 || trainable%: 0.1243
None


In [ ]:
# Настройки обучения
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="epoch",
    push_to_hub=False
)

# Создание тренера
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
   # data_collator=None  # Можно использовать свой коллатор
)

# Обучение модели
trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,13.537800
20,6.693900
30,5.599200


Step,Training Loss
10,13.537800
20,6.693900
30,5.599200
40,5.072400
50,4.643200
60,4.332500


In [ ]:
# Функция для создания запроса
def create_prompt(role, human_input):
    return f"<role>: {role}\n<human>: {human_input}\n<bot>:"

# Входные данные
role = "Я — человек."
human_input = "What is the capital of France?"

# Создание запроса
prompt = create_prompt(role, human_input)

# Токенизация запроса
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Генерация ответа
outputs = model.generate(
    **inputs,
    max_new_tokens=64,  # Максимальная длина ответа
    use_cache=True,
    pad_token_id=tokenizer.eos_token_id  # Для корректной обработки EOS-токена
)

# Декодирование ответа
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Вывод результата
print(response)

In [ ]:
from google.colab import drive

# Подключение Google Drive
drive.mount('/content/drive')

# Путь к Google Drive
output_dir = "/content/drive/MyDrive/lora_finetuned_model"

# Сохранение модели
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Модель успешно сохранена в {output_dir}")